# H4a Nastawienie wobec obozu (NLI)

## 1. Konfiguracja

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/Licencjat/h4a

import sys
sys.path.append("..")

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.config import FINETUNED_MODEL_DIR, PROCESSED_DIR
from src.nli import nli_probabilities

H4A_DIR = PROCESSED_DIR / "h4a"
BATCH_SIZE = 8
device = "cuda" if torch.cuda.is_available() else "cpu"
print("urządzenie:", device)

## 2. Wczytanie wzmianek i modelu

In [ ]:
mentions = pd.read_parquet(H4A_DIR / "mentions.parquet", columns=["id", "subreddit", "cycle", "term", "text"])
tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(FINETUNED_MODEL_DIR, dtype=torch.float32).to(device).eval()
mentions.shape

## 3. Klasyfikacja w pętli po subreddit × cycle

In [ ]:
for (subreddit, cycle), group in tqdm(mentions.groupby(["subreddit", "cycle"])):
    group = group.iloc[group["text"].str.len().argsort().values]
    stance = nli_probabilities(group["text"].tolist(), group["term"].tolist(), model, tokenizer, device, BATCH_SIZE)
    stance.insert(0, "id", group["id"].values)
    stance.to_parquet(H4A_DIR / f"stance_{subreddit}_{cycle}.parquet", index=False)

## 4. Kontrola

In [ ]:
stance = pd.concat(pd.read_parquet(path) for path in sorted(H4A_DIR.glob("stance_*.parquet")))
print(len(stance), len(mentions))
stance[["p_entail", "p_entail_3"]].describe()